# DPO Polish: Preference-Based Reasoning Quality Improvement

This notebook applies **Iterative DPO** (Direct Preference Optimization) to polish
reasoning quality after AdaSTaR self-improvement.

**Training pipeline stage:** 5 of 5 (SFT -> GSPO (curriculum) -> RAFT++ -> AdaSTaR -> **DPO**)

**Target hardware:** Google Colab A100 40GB / 80GB

**Key features:**
- **RAFT++ negative reuse (arXiv 2505.24850):** Loads pre-saved incorrect completions from RAFT++
  as "rejected" samples, eliminating redundant generation. Falls back to self-generation if unavailable.
- Generate preference pairs: chosen (correct + best format) vs rejected (RAFT++ negatives or worst format)
- DPO training with beta=0.1 for conservative policy update
- Guard: skip DPO if < MIN_PAIRS preference pairs (model already strong)
- Dual evaluation: correctness must stay within 1%, format score must improve 10%+

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

**References:**
- [DPO (arXiv 2305.18290)](https://arxiv.org/abs/2305.18290) — Direct Preference Optimization
- [Harnessing Negative Signals (arXiv 2505.24850)](https://arxiv.org/abs/2505.24850) — Reusing RAFT++ negatives
- [Iterative DPO (arXiv 2503.12854)](https://arxiv.org/abs/2503.12854) — Self-play preference learning
- [SimPO (arXiv 2405.14734)](https://arxiv.org/abs/2405.14734) — Length-normalized reference-free DPO

In [ ]:
# Disable gradient offloading BEFORE importing unsloth
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"

# Install dependencies — TRL >= 0.27.0 pinned for consistency across pipeline
!pip install -q unsloth "trl>=0.27.0" peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy

from huggingface_hub import login
login()

In [ ]:
# ============================================================
# Add Drive root to sys.path (training/ is at MyDrive level)
# ============================================================
import sys, os

DRIVE_ROOT = "/content/drive/MyDrive"
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)

for s in ["training/scripts/stem_rewards.py", "training/scripts/verify_answers.py", "training/scripts/sort_curriculum.py"]:
    print(f"  {'OK' if os.path.exists(os.path.join(DRIVE_ROOT, s)) else 'MISSING'} {s}")

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model — Instruct base gives dialogue abilities built-in
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
STAR_CHECKPOINT = "/content/drive/MyDrive/MITS/checkpoints/star_qwen3_4b/final_adapter"
STAR_HF_REPO = "Siesher/mits-qwen3-4b-star"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/dpo_qwen3_4b"

# ---- RAFT++ Negatives (arXiv 2505.24850) ----
# Pre-generated incorrect completions from RAFT++ stage, used as "rejected" samples.
# Eliminates redundant generation — DPO only needs to generate "chosen" (correct) samples.
RAFT_NEGATIVES_PATH = "/content/drive/MyDrive/MITS/checkpoints/raft_qwen3_4b/raft_negatives.jsonl"

# ---- A100 GPU preset ----
A100_VRAM_GB = 40

# DPO parameters
DPO_BETA = 0.1                 # Conservative policy update
DPO_LR = 5e-7                  # Very low LR for polish
DPO_STEPS = 200
DPO_WARMUP_RATIO = 0.1

if A100_VRAM_GB >= 80:
    PAIRS_PER_PROBLEM = 8      # Completions per problem for pair generation
    DPO_BATCH_SIZE = 2
else:
    PAIRS_PER_PROBLEM = 8
    DPO_BATCH_SIZE = 1

DPO_GRAD_ACCUM = 4
MIN_PAIRS = 50                 # Skip DPO if too few valid pairs

# Generation
MAX_COMPLETION = 1024
MAX_PROMPT_LENGTH = 512
MAX_SEQ_LENGTH = MAX_PROMPT_LENGTH + MAX_COMPLETION

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]
SYSTEM_PROMPT = "Ты — репетитор по STEM. Реши задачу пошагово и запиши финальный ответ в \\boxed{}."

print(f"Hardware: A100 {A100_VRAM_GB}GB")
print(f"Base model: {BASE_MODEL}")
print(f"DPO: beta={DPO_BETA}, lr={DPO_LR}, steps={DPO_STEPS}")
print(f"Pairs per problem: {PAIRS_PER_PROBLEM}, min pairs: {MIN_PAIRS}")
print(f"RAFT++ negatives: {RAFT_NEGATIVES_PATH}")

In [ ]:
# ============================================================
# Mount Drive, resolve checkpoint, load data
# ============================================================
import json
import os
import re
import random
from collections import Counter, defaultdict

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/dpo_qwen3_4b"

# Resolve AdaSTaR checkpoint: Drive first, then download from HuggingFace
if os.path.exists(STAR_CHECKPOINT):
    print(f"AdaSTaR checkpoint found: {STAR_CHECKPOINT}")
elif STAR_HF_REPO:
    from huggingface_hub import snapshot_download
    STAR_CHECKPOINT = snapshot_download(STAR_HF_REPO)
    print(f"Downloaded AdaSTaR adapter from HuggingFace to: {STAR_CHECKPOINT}")
else:
    raise FileNotFoundError(f"AdaSTaR checkpoint not found: {STAR_CHECKPOINT}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Load RL problems: Drive JSONL → HF "rl" → HF "gspo" fallback ----
from datasets import load_dataset

RL_DATA_PATH = "/content/drive/MyDrive/training/data/rl_combined.jsonl"

if os.path.exists(RL_DATA_PATH):
    print(f"Loading RL data from Drive: {RL_DATA_PATH}")
    problems = []
    with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            problems.append(json.loads(line))
    print(f"Loaded {len(problems)} from Drive JSONL")
else:
    try:
        print("Trying HF 'rl' config...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "rl")
        problems = [dict(r) for r in hf_ds["train"]]
        if "test" in hf_ds:
            problems += [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} from HF 'rl' config")
    except Exception:
        print("Falling back to HF 'gspo' config...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")
        problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} from HF 'gspo' (fallback)")

verifiable_problems = [
    p for p in problems
    if p.get("type", "verifiable") == "verifiable"
    and p.get("answer_type", "numeric") != "conceptual"
]
print(f"Loaded {len(verifiable_problems)} verifiable problems")

In [ ]:
# ============================================================
# Load model + AdaSTaR adapter
# ============================================================
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

# Load adapter config
star_config_path = os.path.join(STAR_CHECKPOINT, "adapter_config.json")
if os.path.exists(star_config_path):
    with open(star_config_path) as f:
        star_cfg = json.load(f)
    star_r = star_cfg.get("r", LORA_R)
    star_alpha = star_cfg.get("lora_alpha", LORA_ALPHA)
else:
    star_r = LORA_R
    star_alpha = LORA_ALPHA

effective_r = max(star_r, LORA_R)

# Fresh LoRA for DPO
model = FastLanguageModel.get_peft_model(
    model,
    r=effective_r,
    lora_alpha=star_alpha,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Load AdaSTaR weights
from safetensors.torch import load_file
star_weights_path = os.path.join(STAR_CHECKPOINT, "adapter_model.safetensors")
if os.path.exists(star_weights_path):
    star_weights = load_file(star_weights_path)
    incompatible = model.load_state_dict(star_weights, strict=False)
    print(f"Loaded AdaSTaR weights: {len(incompatible.missing_keys)} missing keys")
else:
    print(f"WARNING: AdaSTaR weights not found at {star_weights_path}")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

In [ ]:
# ============================================================
# Verification + format scoring
# ============================================================

_verify_imported = False
try:
    from training.scripts.verify_answers import verify, extract_answer
    _verify_imported = True
    print("Imported verify functions")
except ImportError:
    try:
        import sys
        sys.path.insert(0, "/content/drive/MyDrive")
        from training.scripts.verify_answers import verify, extract_answer
        _verify_imported = True
    except ImportError:
        import sympy
        def extract_answer(text):
            if "</think>" in text: text = text.split("</think>")[-1].strip()
            boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
            return boxed[-1].strip() if boxed else text.strip()


def verify_completion(completion, problem):
    answer = extract_answer(completion)
    truth = problem.get("ground_truth", problem.get("answer", ""))
    domain = problem.get("domain", "math")
    if _verify_imported:
        result = verify(answer=answer, truth=truth, domain=domain,
                       question_type=problem.get("type", "calc"),
                       test_cases=problem.get("test_cases"))
        return result.correct
    return answer.strip().lower() == str(truth).strip().lower()


def score_format(text):
    """Score reasoning format quality [0, 1]."""
    score = 0.0
    if "\\boxed{" in text: score += 0.4
    step_markers = ["step", "therefore", "thus", "hence", "because",
                    "\u0448\u0430\u0433", "\u0441\u043b\u0435\u0434\u043e\u0432\u0430\u0442\u0435\u043b\u044c\u043d\u043e",
                    "\u0437\u043d\u0430\u0447\u0438\u0442", "\u043f\u043e\u0442\u043e\u043c\u0443 \u0447\u0442\u043e",
                    "\u0434\u0430\u043b\u0435\u0435", "\u043f\u043e\u0434\u0441\u0442\u0430\u0432\u0438\u043c"]
    if any(m in text.lower() for m in step_markers): score += 0.3
    word_count = len(text.split())
    if 50 < word_count < 800: score += 0.2
    if "<think>" in text and "</think>" in text: score += 0.1
    return min(1.0, score)


print("Verification and format scoring ready")

In [ ]:
# ============================================================
# Build preference pairs: chosen (correct) vs rejected (RAFT++ negatives)
# (arXiv 2505.24850: Harnessing Negative Signals)
# ============================================================

import random
from collections import defaultdict

# ---- Step 1: Load RAFT++ negatives ----
raft_negatives = []
raft_negatives_by_prompt = defaultdict(list)

if os.path.exists(RAFT_NEGATIVES_PATH):
    with open(RAFT_NEGATIVES_PATH, "r", encoding="utf-8") as f:
        for line in f:
            neg = json.loads(line)
            raft_negatives.append(neg)
            raft_negatives_by_prompt[neg["prompt"]].append(neg["completion"])
    print(f"Loaded {len(raft_negatives)} RAFT++ negatives from {RAFT_NEGATIVES_PATH}")
    print(f"  Covering {len(raft_negatives_by_prompt)} unique prompts")
else:
    print(f"RAFT++ negatives not found at {RAFT_NEGATIVES_PATH}")
    print("Will generate rejected samples fresh (slower)")

# ---- Step 2: Generate chosen (correct) completions + pair with rejected ----
print("\nGenerating preference pairs...")
FastLanguageModel.for_inference(model)

preference_pairs = []
pair_stats = {
    "total_problems_attempted": 0,
    "chosen_generated": 0,
    "rejected_from_raft": 0,
    "rejected_generated_fresh": 0,
    "problems_with_pairs": 0,
}

# Sample problems for DPO
dpo_sample_size = min(len(verifiable_problems), 500)
dpo_problems = random.sample(verifiable_problems, dpo_sample_size)

for i, problem in enumerate(dpo_problems):
    pair_stats["total_problems_attempted"] += 1
    domain = problem.get("domain", "math")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem["prompt"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt_text, return_tensors="pt",
        truncation=True, max_length=MAX_PROMPT_LENGTH
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    # Generate PAIRS_PER_PROBLEM completions
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_COMPLETION,
            temperature=0.9, do_sample=True,
            num_return_sequences=PAIRS_PER_PROBLEM,
        )

    # Classify as correct/incorrect + score format
    correct_comps = []
    incorrect_comps = []
    for j in range(outputs.shape[0]):
        comp = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
        is_correct = verify_completion(comp, problem)
        fmt_score = score_format(comp)

        if is_correct:
            correct_comps.append((comp, fmt_score))
        else:
            incorrect_comps.append((comp, fmt_score))

    if not correct_comps:
        continue  # Need at least one correct completion for "chosen"

    # Pick best correct as "chosen" (highest format score)
    correct_comps.sort(key=lambda x: x[1], reverse=True)
    chosen_comp = correct_comps[0][0]
    pair_stats["chosen_generated"] += 1

    # Pick "rejected": prefer RAFT++ negatives, fall back to generated
    rejected_comp = None

    # Try RAFT++ negatives first
    raft_negs = raft_negatives_by_prompt.get(problem["prompt"], [])
    if raft_negs:
        rejected_comp = random.choice(raft_negs)
        pair_stats["rejected_from_raft"] += 1
    elif incorrect_comps:
        # Use worst generated incorrect completion
        incorrect_comps.sort(key=lambda x: x[1])
        rejected_comp = incorrect_comps[0][0]
        pair_stats["rejected_generated_fresh"] += 1

    if rejected_comp is None:
        continue  # No rejected sample available

    pair_stats["problems_with_pairs"] += 1

    # Format as DPO preference pair
    chosen_messages = messages + [{"role": "assistant", "content": chosen_comp}]
    rejected_messages = messages + [{"role": "assistant", "content": rejected_comp}]

    preference_pairs.append({
        "prompt": prompt_text,
        "chosen": tokenizer.apply_chat_template(
            chosen_messages, tokenize=False, add_generation_prompt=False
        ),
        "rejected": tokenizer.apply_chat_template(
            rejected_messages, tokenize=False, add_generation_prompt=False
        ),
    })

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{dpo_sample_size}: {len(preference_pairs)} pairs")

FastLanguageModel.for_training(model)

dpo_loss = None  # Will be set during training

print(f"\nPreference pair generation complete:")
print(f"  Total pairs: {len(preference_pairs)}")
print(f"  Rejected from RAFT++ negatives: {pair_stats['rejected_from_raft']}")
print(f"  Rejected generated fresh: {pair_stats['rejected_generated_fresh']}")
print(f"  Min pairs threshold: {MIN_PAIRS}")

In [ ]:
# ============================================================
# DPO Training (with guard + Colab disconnect recovery)
# ============================================================
from trl import DPOConfig, DPOTrainer
from datasets import Dataset


def find_latest_checkpoint(output_dir):
    """Find latest TRL checkpoint for resume after Colab disconnect."""
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    path = os.path.join(output_dir, latest)
    print(f"  Found checkpoint: {path}")
    return path


dpo_skipped = False

if len(preference_pairs) < MIN_PAIRS:
    print(f"\nWARNING: Only {len(preference_pairs)} pairs < MIN_PAIRS={MIN_PAIRS}")
    print("Skipping DPO — AdaSTaR checkpoint is already strong enough.")
    print("The AdaSTaR adapter will be used as the final model.")
    dpo_skipped = True
else:
    print(f"\nProceeding with DPO training on {len(preference_pairs)} preference pairs")

    # Format as HuggingFace Dataset
    dpo_dataset = Dataset.from_list(preference_pairs)
    print(f"DPO dataset: {len(dpo_dataset)} examples")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    dpo_config = DPOConfig(
        output_dir=OUTPUT_DIR,
        max_steps=DPO_STEPS,
        per_device_train_batch_size=DPO_BATCH_SIZE,
        gradient_accumulation_steps=DPO_GRAD_ACCUM,
        learning_rate=DPO_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=DPO_WARMUP_RATIO,
        beta=DPO_BETA,
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
        bf16=True,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        optim="adamw_torch_fused",
        seed=42,
        report_to="none",
    )

    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
    )

    # Resume from checkpoint if Colab session was interrupted
    dpo_resume = find_latest_checkpoint(OUTPUT_DIR)
    print("Starting DPO training...")
    result = trainer.train(resume_from_checkpoint=dpo_resume)
    dpo_loss = result.training_loss
    print(f"DPO training complete! Loss: {dpo_loss:.4f}")

    trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
    del trainer
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Dual evaluation: correctness + format quality
# ============================================================

print("Running dual evaluation...")
FastLanguageModel.for_inference(model)

eval_size = min(80, len(verifiable_problems))
eval_problems = random.sample(verifiable_problems, eval_size)

domain_results = defaultdict(lambda: {"total": 0, "correct": 0, "format_scores": []})

for problem in eval_problems:
    domain = problem.get("domain", "math")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem["prompt"]},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_COMPLETION, temperature=0.7, do_sample=True)
    comp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    domain_results[domain]["total"] += 1
    if verify_completion(comp, problem):
        domain_results[domain]["correct"] += 1
    domain_results[domain]["format_scores"].append(score_format(comp))

# Print results
print(f"\nFinal DPO evaluation:")
total_c, total_t = 0, 0
all_fmt = []
final_domain_metrics = {}
for domain in DOMAINS:
    s = domain_results[domain]
    if s["total"] > 0:
        acc = 100 * s["correct"] / s["total"]
        mean_fmt = sum(s["format_scores"]) / len(s["format_scores"])
        print(f"  {domain}: acc={acc:.1f}%, format={mean_fmt:.3f} ({s['total']} problems)")
        final_domain_metrics[domain] = {
            "accuracy": s["correct"] / s["total"],
            "mean_format": mean_fmt,
            "total": s["total"],
        }
        total_c += s["correct"]
        total_t += s["total"]
        all_fmt.extend(s["format_scores"])

final_acc = total_c / total_t if total_t > 0 else 0
final_fmt = sum(all_fmt) / len(all_fmt) if all_fmt else 0
print(f"\n  Overall accuracy: {100*final_acc:.1f}%")
print(f"  Overall format score: {final_fmt:.3f}")
print(f"  DPO skipped: {dpo_skipped}")

In [ ]:
# ============================================================
# Save final adapter + metrics
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final DPO adapter saved to {final_adapter_path}")

# Save metrics
eval_metrics = {
    "stage": "dpo" if not dpo_skipped else "dpo_skipped",
    "pipeline_position": "5 of 5",
    "dpo_skipped": dpo_skipped,
    "final_accuracy": final_acc,
    "final_format_score": final_fmt,
    "domain_results": final_domain_metrics,
    "preference_pairs_generated": len(preference_pairs),
    "pair_generation_stats": pair_stats,
    "dpo_loss": dpo_loss if not dpo_skipped else None,
}
eval_path = os.path.join(OUTPUT_DIR, "dpo_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump(eval_metrics, f, indent=2, default=str)
print(f"Eval metrics saved to {eval_path}")

# Save config
config = {
    "stage": "dpo",
    "pipeline": "SFT -> GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO",
    "pipeline_position": "5 of 5",
    "base_model": BASE_MODEL,
    "star_checkpoint": STAR_CHECKPOINT,
    "hardware": f"A100 {A100_VRAM_GB}GB",
    "dpo_beta": DPO_BETA,
    "dpo_lr": DPO_LR,
    "dpo_steps": DPO_STEPS,
    "pairs_per_problem": PAIRS_PER_PROBLEM,
    "min_pairs": MIN_PAIRS,
    "dpo_skipped": dpo_skipped,
    "lora_r": effective_r,
    "lora_alpha": star_alpha,
    "lora_dropout": LORA_DROPOUT,
    "total_problems": len(verifiable_problems),
    "preference_pairs": len(preference_pairs),
    "references": [
        "DPO arXiv:2305.18290",
        "Iterative DPO arXiv:2503.12854",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {config_path}")

# Optional: push to HF
PUSH_TO_HUB = False
HF_REPO_ID = "Siesher/mits-qwen3-4b-final"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\n" + "="*60)
print("TRAINING PIPELINE COMPLETE")
print("="*60)
print(f"Pipeline: SFT -> GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO")
print(f"Final model: {final_adapter_path}")
print(f"Overall accuracy: {100*final_acc:.1f}%")
print(f"Format quality: {final_fmt:.3f}")
print("\nThe final adapter can be exported to GGUF for Ollama deployment.")